# 02 · Capa Silver — Transformación y calidad de datos

**IDY1102 · EP3 · DataRetail 360 · Proceso Ventas**

| Campo | Detalle |
|---|---|
| **Capa Medallion** | Silver (datos limpios, validados y estandarizados) |
| **Etapa ETL** | TRANSFORM (sección 2 de la plantilla 2.2.3) |
| **IL** | IL 2.3 (validaciones y manejo de errores) · IL 2.4 (datos limpios y trazables) |
| **Entrada** | Tablas `bronze_*` del lote |
| **Salida** | Tablas `silver_*` + tabla de incidencias `etl_incidencias` |

**Qué hace esta capa**
1. **Convierte tipos**: el texto de Bronze pasa a números y fechas.
2. **Aplica reglas de calidad**, cada una asociada a una dimensión de calidad (unicidad, validez, completitud, consistencia, integridad).
3. **Unifica** las ventas de tienda y online en un solo formato.
4. **Protege datos personales** (Ley 21.719): elimina nombre y correo, y seudonimiza el identificador del cliente.

**Qué pasa con los errores:** ninguna fila se pierde en silencio. Cada problema queda en `etl_incidencias` con su regla, su dimensión de calidad y la acción tomada:
- **Rechazada**: la fila no pasa a Silver (por ejemplo, una fecha imposible).
- **Corregida**: la fila pasa a Silver con el valor arreglado (por ejemplo, una comuna mal escrita).

## 0. Configuración

In [ ]:
import hashlib
import os
import sys
import unicodedata
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

RAIZ = Path.cwd()
while not (RAIZ / "src" / "config.py").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
sys.path.append(str(RAIZ / "src"))

import config
import utils

# >>> Lote a procesar (1 = enero-junio, 2 = julio-septiembre) <<<
ID_LOTE = 2

logger = utils.configurar_logging("silver")
motor = utils.obtener_motor()

# La "sal" de seudonimización es un secreto: se lee del archivo .env, nunca va en el código
load_dotenv(RAIZ / ".env")
SAL = os.getenv("SAL_SEUDONIMIZACION")
if not SAL:
    logger.error("Falta SAL_SEUDONIMIZACION en el archivo .env. Se detiene el pipeline.")
    raise RuntimeError("Configure SAL_SEUDONIMIZACION en .env (ver .env.example)")

logger.info(f"=== Inicio capa Silver | lote {ID_LOTE} ===")

2026-10-25 21:19:29 | INFO    | silver | === Inicio capa Silver | lote 1 ===


## 1. Leer Bronze

Se leen solo las filas del lote que se está procesando.

In [2]:
def leer_bronze(fuente: str) -> pd.DataFrame | None:
    """Lee bronze_<fuente> para el lote actual. Devuelve None si la fuente no vino en este lote."""
    tabla = f"bronze_{fuente}"
    if not utils.tabla_existe(motor, tabla):
        return None
    df = pd.read_sql(f"SELECT * FROM {tabla} WHERE _id_lote = {ID_LOTE}", motor)
    return df if len(df) > 0 else None


bronze = {fuente: leer_bronze(fuente) for fuente in config.FUENTES_POR_LOTE[ID_LOTE]}
for fuente, df in bronze.items():
    logger.info(f"Leído bronze_{fuente}: {0 if df is None else len(df)} filas")

2026-10-25 21:19:29 | INFO    | silver | Leído bronze_canales: 3 filas
2026-10-25 21:19:29 | INFO    | silver | Leído bronze_sucursales: 10 filas
2026-10-25 21:19:29 | INFO    | silver | Leído bronze_campanas: 5 filas
2026-10-25 21:19:29 | INFO    | silver | Leído bronze_productos: 32 filas
2026-10-25 21:19:29 | INFO    | silver | Leído bronze_clientes: 405 filas
2026-10-25 21:19:29 | INFO    | silver | Leído bronze_ventas_tienda: 7452 filas
2026-10-25 21:19:29 | INFO    | silver | Leído bronze_ventas_online: 2367 filas


## 2. Herramientas de calidad

Dos funciones que usan todas las reglas:
- `registrar`: anota incidencias en la lista `incidencias`.
- `separar`: divide un DataFrame en filas válidas y filas rechazadas según una condición, y anota las rechazadas.

In [3]:
incidencias = []   # aquí se acumulan todas las incidencias del lote


def registrar(df_afectado: pd.DataFrame, fuente: str, col_registro: str, col_linea: str | None,
              regla: str, dimension: str, motivo: str, accion: str) -> None:
    """Anota cada fila de df_afectado como una incidencia de calidad."""
    for _, fila in df_afectado.iterrows():
        incidencias.append({
            "_id_lote": ID_LOTE,
            "fuente": fuente,
            "registro": fila[col_registro],
            "linea": int(fila[col_linea]) if col_linea else 0,
            "fila_origen": int(fila["_fila_origen"]),
            "regla": regla,
            "dimension_calidad": dimension,
            "motivo": motivo,
            "accion": accion,
        })


def separar(df: pd.DataFrame, invalidas: pd.Series, fuente: str, col_registro: str,
            col_linea: str | None, regla: str, dimension: str, motivo: str) -> pd.DataFrame:
    """Rechaza las filas marcadas en 'invalidas' y devuelve solo las válidas."""
    rechazadas = df[invalidas]
    if len(rechazadas) > 0:
        registrar(rechazadas, fuente, col_registro, col_linea, regla, dimension, motivo, "Rechazada")
        logger.warning(f"[{regla}] {fuente}: {len(rechazadas)} filas rechazadas ({motivo})")
    return df[~invalidas].copy()


def sin_tildes(texto: str) -> str:
    """'  Ñuñoa ' -> 'nunoa'. Sirve para comparar textos escritos de distintas formas."""
    texto = unicodedata.normalize("NFKD", str(texto).strip().lower())
    return "".join(c for c in texto if not unicodedata.combining(c))


def seudonimizar(id_cliente: str) -> str:
    """Reemplaza el ID real por un código irreversible (SHA-256 con sal)."""
    return hashlib.sha256((SAL + id_cliente).encode("utf-8")).hexdigest()[:16]


TRAZA = ["_id_lote", "_archivo_origen", "_fila_origen"]   # columnas de trazabilidad que se conservan

## 3. Catálogos: canales, sucursales y campañas

Solo vienen en el lote 1. Se convierten los tipos de datos (validez).

In [4]:
silver = {}   # aquí se guardan las tablas Silver listas para cargar

if bronze.get("canales") is not None:
    silver["canales"] = bronze["canales"][["cod_canal", "nombre_canal", "tipo_canal"] + TRAZA]

if bronze.get("sucursales") is not None:
    suc = bronze["sucursales"].copy()
    suc["latitud"] = pd.to_numeric(suc["latitud"], errors="coerce")     # vacío -> nulo (tienda online)
    suc["longitud"] = pd.to_numeric(suc["longitud"], errors="coerce")
    suc["fecha_apertura"] = pd.to_datetime(suc["fecha_apertura"], format="%Y-%m-%d").dt.strftime("%Y-%m-%d")
    silver["sucursales"] = suc

if bronze.get("campanas") is not None:
    cam = bronze["campanas"].copy()
    cam["presupuesto"] = pd.to_numeric(cam["presupuesto"]).astype("int64")
    silver["campanas"] = cam

logger.info(f"Catálogos procesados: {[c for c in ('canales', 'sucursales', 'campanas') if c in silver]}")

2026-10-25 21:19:29 | INFO    | silver | Catálogos procesados: ['canales', 'sucursales', 'campanas']


## 4. Productos

| Regla | Dimensión | Acción |
|---|---|---|
| P1 · Precio y costo deben ser números positivos | Validez | Rechazar |
| P2 · Código de producto único | Unicidad | Rechazar duplicados |

In [5]:
prod = bronze["productos"].copy()
prod["precio_lista"] = pd.to_numeric(prod["precio_lista"], errors="coerce")
prod["costo_unitario"] = pd.to_numeric(prod["costo_unitario"], errors="coerce")

prod = separar(prod, ~((prod["precio_lista"] > 0) & (prod["costo_unitario"] > 0)),
               "productos", "cod_producto", None, "P1", "Validez", "Precio o costo no válido")
prod = separar(prod, prod.duplicated("cod_producto", keep="first"),
               "productos", "cod_producto", None, "P2", "Unicidad", "Producto duplicado")

prod["precio_lista"] = prod["precio_lista"].astype("int64")
prod["costo_unitario"] = prod["costo_unitario"].astype("int64")
for col in ("nombre_producto", "marca", "subcategoria", "categoria", "estado"):
    prod[col] = prod[col].str.strip()

silver["productos"] = prod
logger.info(f"Productos válidos: {len(prod)}")

2026-10-25 21:19:29 | INFO    | silver | Productos válidos: 32


## 5. Clientes

| Regla | Dimensión | Acción |
|---|---|---|
| C1 · Un registro por cliente | Unicidad | Rechazar duplicados |
| C2 · Comuna escrita según la tabla oficial | Consistencia | Corregir |
| C3 · Comuna inexistente | Validez | Rechazar |
| C4 · La región debe corresponder a la comuna (jerarquía) | Consistencia | Corregir |
| C5 · Fecha de nacimiento faltante | Completitud | Corregir: tramo "Sin información" |

**Protección de datos (Ley 21.719):** al final se eliminan `nombre` y `email` (minimización) y el `id_cliente` se reemplaza por un código seudonimizado. En la tabla de incidencias también se guarda el código seudonimizado, no el ID real.

In [6]:
cli = bronze["clientes"].copy()
cli["id_seud"] = cli["id_cliente"].map(seudonimizar)

# C1 · Unicidad
cli = separar(cli, cli.duplicated("id_cliente", keep="first"),
              "clientes", "id_seud", None, "C1", "Unicidad", "Cliente duplicado")

# C2 · Consistencia: homologar la comuna con la tabla oficial
comuna_oficial = {sin_tildes(c): c for c in config.REGION_POR_COMUNA}
cli["comuna_homologada"] = cli["comuna"].map(lambda c: comuna_oficial.get(sin_tildes(c)))
corregidas = cli[cli["comuna_homologada"].notna() & (cli["comuna"] != cli["comuna_homologada"])]
registrar(corregidas, "clientes", "id_seud", None, "C2", "Consistencia",
          "Comuna con formato inconsistente", "Corregida")
logger.warning(f"[C2] clientes: {len(corregidas)} comunas homologadas")

# C3 · Validez: comuna que no existe en la tabla oficial
cli = separar(cli, cli["comuna_homologada"].isna(),
              "clientes", "id_seud", None, "C3", "Validez", "Comuna inexistente")
cli["comuna"] = cli["comuna_homologada"]

# C4 · Consistencia jerárquica: región según la tabla oficial
cli["region_oficial"] = cli["comuna"].map(config.REGION_POR_COMUNA)
mal_region = cli[cli["region"] != cli["region_oficial"]]
registrar(mal_region, "clientes", "id_seud", None, "C4", "Consistencia",
          "Región no coincide con la comuna", "Corregida")
logger.warning(f"[C4] clientes: {len(mal_region)} regiones corregidas según la comuna")
cli["region"] = cli["region_oficial"]

# C5 · Completitud: fecha de nacimiento -> tramo de edad
cli["fecha_nacimiento"] = pd.to_datetime(cli["fecha_nacimiento"], format="%Y-%m-%d", errors="coerce")
sin_fecha = cli[cli["fecha_nacimiento"].isna()]
registrar(sin_fecha, "clientes", "id_seud", None, "C5", "Completitud",
          "Fecha de nacimiento vacía", "Corregida")
logger.warning(f"[C5] clientes: {len(sin_fecha)} sin fecha de nacimiento -> 'Sin información'")

fecha_corte = pd.Timestamp(config.FECHA_CORTE_POR_LOTE[ID_LOTE])
edad = (fecha_corte - cli["fecha_nacimiento"]).dt.days // 365


def tramo(e) -> str:
    if pd.isna(e):
        return "Sin información"
    for minimo, maximo, etiqueta in config.TRAMOS_EDAD:
        if minimo <= e <= maximo:
            return etiqueta
    return "Sin información"


cli["tramo_edad"] = edad.map(tramo)

# Minimización y seudonimización: solo pasan a Silver los atributos que el análisis necesita
mapa_clientes = dict(zip(cli["id_cliente"], cli["id_seud"]))   # ID real -> seudónimo (solo en memoria)
silver["clientes"] = (cli.rename(columns={"id_seud": "id_cliente_seud"})
                         [["id_cliente_seud", "segmento", "tramo_edad", "comuna", "region",
                           "fecha_registro"] + TRAZA])
logger.info(f"Clientes válidos: {len(silver['clientes'])} (sin nombre ni email)")

2026-10-25 21:19:29 | WARNING | silver | [C1] clientes: 5 filas rechazadas (Cliente duplicado)
2026-10-25 21:19:29 | WARNING | silver | [C2] clientes: 5 comunas homologadas
2026-10-25 21:19:29 | WARNING | silver | [C4] clientes: 5 regiones corregidas según la comuna
2026-10-25 21:19:29 | WARNING | silver | [C5] clientes: 5 sin fecha de nacimiento -> 'Sin información'
2026-10-25 21:19:29 | INFO    | silver | Clientes válidos: 400 (sin nombre ni email)


## 6. Ventas de tienda (POS)

| Regla | Dimensión | Acción |
|---|---|---|
| V1 · Una fila por boleta + línea | Unicidad | Rechazar duplicados |
| V2 · Fecha válida (AAAA-MM-DD) | Validez | Rechazar |
| V3 · Cantidad mayor que cero | Validez | Rechazar |
| V4 · Precio informado | Completitud | Rechazar |
| V5 · El producto existe en el maestro | Integridad referencial | Rechazar |
| V6 · El cliente (si viene) existe en el maestro | Integridad referencial | Rechazar |

**Deduplicación por llave de negocio:** a diferencia de la plantilla 2.2.3 (que compara fecha + producto + sucursal), aquí un duplicado es la misma `nro_boleta` + `nro_linea`. Así no se eliminan ventas reales de clientes distintos que compran lo mismo el mismo día.

In [7]:
vt = bronze["ventas_tienda"].copy()
F = "ventas_tienda"

vt = separar(vt, vt.duplicated(["nro_boleta", "nro_linea"], keep="first"),
             F, "nro_boleta", "nro_linea", "V1", "Unicidad", "Línea duplicada")

vt["fecha"] = pd.to_datetime(vt["fecha"], format="%Y-%m-%d", errors="coerce")
vt = separar(vt, vt["fecha"].isna(), F, "nro_boleta", "nro_linea", "V2", "Validez", "Fecha inválida")

vt["cantidad"] = pd.to_numeric(vt["cantidad"], errors="coerce")
vt = separar(vt, ~(vt["cantidad"] > 0), F, "nro_boleta", "nro_linea", "V3", "Validez", "Cantidad no válida")

vt["precio_unitario"] = pd.to_numeric(vt["precio_unitario"], errors="coerce")
vt = separar(vt, vt["precio_unitario"].isna(), F, "nro_boleta", "nro_linea", "V4", "Completitud", "Precio vacío")

vt = separar(vt, ~vt["cod_producto"].isin(silver["productos"]["cod_producto"]),
             F, "nro_boleta", "nro_linea", "V5", "Integridad referencial", "Producto inexistente")

vt = separar(vt, (vt["id_cliente"] != "") & ~vt["id_cliente"].isin(mapa_clientes.keys()),
             F, "nro_boleta", "nro_linea", "V6", "Integridad referencial", "Cliente inexistente")

# Llevar al formato unificado
vt_unif = pd.DataFrame({
    "origen": "POS",
    "nro_documento": vt["nro_boleta"],
    "nro_linea": vt["nro_linea"].astype("int64"),
    "fecha": vt["fecha"].dt.strftime("%Y-%m-%d"),
    "cod_sucursal": vt["cod_sucursal"],
    "cod_canal": config.CANAL_TIENDA,
    "id_cliente_seud": vt["id_cliente"].map(mapa_clientes),        # vacío -> nulo (no identificado)
    "cod_producto": vt["cod_producto"],
    "cod_campana": vt["cod_campana"].replace("", None),
    "cantidad": vt["cantidad"].astype("int64"),
    "precio_unitario": vt["precio_unitario"].astype("int64"),
    "descuento": pd.to_numeric(vt["descuento"]).astype("int64"),
    "es_devolucion": (vt["tipo_documento"] == "NOTA_CREDITO").astype("int64"),
    "es_anulada": (vt["estado"] == "ANULADA").astype("int64"),
})
vt_unif[TRAZA] = vt[TRAZA]
logger.info(f"Ventas tienda válidas: {len(vt_unif)}")

2026-10-25 21:19:29 | WARNING | silver | [V1] ventas_tienda: 8 filas rechazadas (Línea duplicada)
2026-10-25 21:19:29 | WARNING | silver | [V2] ventas_tienda: 8 filas rechazadas (Fecha inválida)
2026-10-25 21:19:29 | WARNING | silver | [V3] ventas_tienda: 8 filas rechazadas (Cantidad no válida)
2026-10-25 21:19:29 | WARNING | silver | [V4] ventas_tienda: 8 filas rechazadas (Precio vacío)
2026-10-25 21:19:29 | WARNING | silver | [V5] ventas_tienda: 8 filas rechazadas (Producto inexistente)
2026-10-25 21:19:29 | INFO    | silver | Ventas tienda válidas: 7412


## 7. Ventas online (e-commerce)

Además de las reglas de calidad, aquí se **homologa** el formato distinto de esta fuente:

| Origen e-commerce | Formato unificado |
|---|---|
| `id_pedido`, `item`, `sku`, `unidades` | `nro_documento`, `nro_linea`, `cod_producto`, `cantidad` |
| Fecha `DD/MM/AAAA HH:MM` | Fecha `AAAA-MM-DD` |
| Descuento en % (`dscto_pct`) | Descuento en pesos |
| Canal `WEB` / `APP` | Códigos `CAN02` / `CAN03` |
| Estado `DEVUELTO` / `CANCELADO` | Indicadores `es_devolucion` / `es_anulada` |

In [8]:
vo = bronze["ventas_online"].copy()
F = "ventas_online"

vo = separar(vo, vo.duplicated(["id_pedido", "item"], keep="first"),
             F, "id_pedido", "item", "V1", "Unicidad", "Línea duplicada")

vo["fecha"] = pd.to_datetime(vo["fecha_pedido"], format="%d/%m/%Y %H:%M", errors="coerce")
vo = separar(vo, vo["fecha"].isna(), F, "id_pedido", "item", "V2", "Validez", "Fecha inválida")

vo["unidades"] = pd.to_numeric(vo["unidades"], errors="coerce")
vo = separar(vo, ~(vo["unidades"] > 0), F, "id_pedido", "item", "V3", "Validez", "Cantidad no válida")

vo["precio"] = pd.to_numeric(vo["precio"], errors="coerce")
vo = separar(vo, vo["precio"].isna(), F, "id_pedido", "item", "V4", "Completitud", "Precio vacío")

vo = separar(vo, ~vo["sku"].isin(silver["productos"]["cod_producto"]),
             F, "id_pedido", "item", "V5", "Integridad referencial", "Producto inexistente")

vo = separar(vo, ~vo["cliente"].isin(mapa_clientes.keys()),
             F, "id_pedido", "item", "V6", "Integridad referencial", "Cliente inexistente")

pct = pd.to_numeric(vo["dscto_pct"]).fillna(0)
vo_unif = pd.DataFrame({
    "origen": "ECOMMERCE",
    "nro_documento": vo["id_pedido"],
    "nro_linea": vo["item"].astype("int64"),
    "fecha": vo["fecha"].dt.strftime("%Y-%m-%d"),
    "cod_sucursal": config.SUCURSAL_ONLINE,
    "cod_canal": vo["canal"].map(config.CANAL_ONLINE),
    "id_cliente_seud": vo["cliente"].map(mapa_clientes),
    "cod_producto": vo["sku"],
    "cod_campana": vo["campana"].replace("", None),
    "cantidad": vo["unidades"].astype("int64"),
    "precio_unitario": vo["precio"].astype("int64"),
    "descuento": (vo["precio"] * vo["unidades"] * pct / 100).round().astype("int64"),
    "es_devolucion": (vo["estado_pedido"] == "DEVUELTO").astype("int64"),
    "es_anulada": (vo["estado_pedido"] == "CANCELADO").astype("int64"),
})
vo_unif[TRAZA] = vo[TRAZA]
logger.info(f"Ventas online válidas: {len(vo_unif)}")

2026-10-25 21:19:29 | WARNING | silver | [V1] ventas_online: 5 filas rechazadas (Línea duplicada)
2026-10-25 21:19:29 | WARNING | silver | [V3] ventas_online: 5 filas rechazadas (Cantidad no válida)
2026-10-25 21:19:29 | WARNING | silver | [V6] ventas_online: 5 filas rechazadas (Cliente inexistente)
2026-10-25 21:19:29 | INFO    | silver | Ventas online válidas: 2352


## 8. Unificar ventas y calcular medidas

Aquí se aplican las **definiciones oficiales** de las métricas (capa semántica del modelo):

- `monto_bruto = cantidad × precio_unitario`
- `monto_neto = monto_bruto − descuento`
- `costo = cantidad × costo_unitario` (costo del maestro de productos del mismo lote)
- `margen = monto_neto − costo`

**Devoluciones:** sus medidas se guardan con **signo negativo**, para que al sumar la venta neta se descuente lo devuelto. Las ventas **anuladas** se conservan (trazabilidad), marcadas con `es_anulada = 1`; las métricas oficiales las excluirán.

In [9]:
ventas = pd.concat([vt_unif, vo_unif], ignore_index=True)
ventas = ventas.merge(silver["productos"][["cod_producto", "costo_unitario"]], on="cod_producto", how="left")

signo = ventas["es_devolucion"].map({0: 1, 1: -1})
ventas["monto_bruto"] = ventas["cantidad"] * ventas["precio_unitario"]
ventas["monto_neto"] = ventas["monto_bruto"] - ventas["descuento"]
ventas["costo"] = ventas["cantidad"] * ventas["costo_unitario"]
for medida in ("cantidad", "monto_bruto", "descuento", "monto_neto", "costo"):
    ventas[medida] = ventas[medida] * signo
ventas["margen"] = ventas["monto_neto"] - ventas["costo"]
ventas = ventas.drop(columns="costo_unitario")

silver["ventas"] = ventas
logger.info(f"Ventas unificadas: {len(ventas)} filas "
            f"({len(vt_unif)} tienda + {len(vo_unif)} online)")

2026-10-25 21:19:29 | INFO    | silver | Ventas unificadas: 9764 filas (7412 tienda + 2352 online)


## 9. LOAD — cargar Silver e incidencias

Misma estrategia que Bronze: **recarga controlada por lote** dentro de una transacción.

In [10]:
df_incidencias = pd.DataFrame(incidencias)
df_incidencias["fecha_deteccion"] = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")

for nombre, df in silver.items():
    utils.cargar_por_lote(motor, df, f"silver_{nombre}", ID_LOTE, "silver", f"bronze_{nombre}", logger)
_ = utils.cargar_por_lote(motor, df_incidencias, "etl_incidencias", ID_LOTE, "silver", "reglas de calidad", logger)

2026-10-25 21:19:29 | INFO    | silver | Cargado silver_canales: 3 filas
2026-10-25 21:19:29 | INFO    | silver | Cargado silver_sucursales: 10 filas
2026-10-25 21:19:29 | INFO    | silver | Cargado silver_campanas: 5 filas
2026-10-25 21:19:29 | INFO    | silver | Cargado silver_productos: 32 filas
2026-10-25 21:19:29 | INFO    | silver | Cargado silver_clientes: 400 filas
2026-10-25 21:19:30 | INFO    | silver | Cargado silver_ventas: 9764 filas
2026-10-25 21:19:30 | INFO    | silver | Cargado etl_incidencias: 75 filas


## 10. VALIDACIÓN

1. **Conservación de filas:** toda fila de Bronze terminó en Silver o fue rechazada (ninguna se perdió en silencio).
2. **Calidad del resultado:** sin nulos en columnas clave y sin duplicados.
3. **Resumen de incidencias** por regla y dimensión de calidad.

In [11]:
# 1. Conservación de filas: Bronze = Silver + Rechazadas
rechazadas = df_incidencias[df_incidencias["accion"] == "Rechazada"].groupby("fuente").size()
for fuente, n_silver in [("clientes", len(silver["clientes"])), ("productos", len(silver["productos"])),
                         ("ventas_tienda", len(vt_unif)), ("ventas_online", len(vo_unif))]:
    n_bronze = len(bronze[fuente])
    n_rech = int(rechazadas.get(fuente, 0))
    assert n_bronze == n_silver + n_rech, f"{fuente}: se perdieron filas sin registro"
    logger.info(f"Conservación OK {fuente}: {n_bronze} bronze = {n_silver} silver + {n_rech} rechazadas")

# 2. Calidad del resultado
claves = ["nro_documento", "nro_linea", "fecha", "cod_sucursal", "cod_canal", "cod_producto"]
assert ventas[claves].notna().all().all(), "Hay ventas con columnas clave vacías"
assert not ventas.duplicated(["origen", "nro_documento", "nro_linea"]).any(), "Hay ventas duplicadas"
assert silver["clientes"]["id_cliente_seud"].is_unique, "Hay clientes duplicados"
assert "nombre" not in silver["clientes"] and "email" not in silver["clientes"], "Hay datos personales en Silver"
logger.info("Validación de calidad OK")

# 3. Resumen de incidencias
resumen = (df_incidencias.groupby(["fuente", "regla", "dimension_calidad", "motivo", "accion"])
           .size().reset_index(name="filas"))
logger.info(f"Incidencias del lote {ID_LOTE}: {len(df_incidencias)} "
            f"({(df_incidencias['accion'] == 'Rechazada').sum()} rechazadas, "
            f"{(df_incidencias['accion'] == 'Corregida').sum()} corregidas)")
resumen

2026-10-25 21:19:30 | INFO    | silver | Conservación OK clientes: 405 bronze = 400 silver + 5 rechazadas
2026-10-25 21:19:30 | INFO    | silver | Conservación OK productos: 32 bronze = 32 silver + 0 rechazadas
2026-10-25 21:19:30 | INFO    | silver | Conservación OK ventas_tienda: 7452 bronze = 7412 silver + 40 rechazadas
2026-10-25 21:19:30 | INFO    | silver | Conservación OK ventas_online: 2367 bronze = 2352 silver + 15 rechazadas
2026-10-25 21:19:30 | INFO    | silver | Validación de calidad OK
2026-10-25 21:19:30 | INFO    | silver | Incidencias del lote 1: 75 (60 rechazadas, 15 corregidas)


,fuente,regla,dimension_calidad,motivo,accion,filas
0,clientes,C1,Unicidad,Cliente duplicado,Rechazada,5
1,clientes,C2,Consistencia,Comuna con formato inconsistente,Corregida,5
2,clientes,C4,Consistencia,Región no coincide con la comuna,Corregida,5
3,clientes,C5,Completitud,Fecha de nacimiento vacía,Corregida,5
4,ventas_online,V1,Unicidad,Línea duplicada,Rechazada,5
5,ventas_online,V3,Validez,Cantidad no válida,Rechazada,5
6,ventas_online,V6,Integridad referencial,Cliente inexistente,Rechazada,5
7,ventas_tienda,V1,Unicidad,Línea duplicada,Rechazada,8
8,ventas_tienda,V2,Validez,Fecha inválida,Rechazada,8
9,ventas_tienda,V3,Validez,Cantidad no válida,Rechazada,8


## 11. Contraste con los errores inyectados

El generador de datos anotó cada error que introdujo a propósito (`data/errores_inyectados_loteN.csv`). El ETL **no usa ese archivo para detectar nada**: aquí solo se compara al final, como prueba de que las reglas funcionan.

In [12]:
esperados = pd.read_csv(RAIZ / "data" / f"errores_inyectados_lote{ID_LOTE}.csv", dtype=str)
# Los clientes se comparan por su seudónimo, igual que en etl_incidencias
es_cliente = esperados["archivo"] == "clientes"
esperados.loc[es_cliente, "registro"] = esperados.loc[es_cliente, "registro"].map(seudonimizar)

clave = lambda df, col_fuente: set(zip(df[col_fuente], df["registro"].astype(str), df["linea"].astype(int)))
set_esperado = clave(esperados, "archivo")
set_detectado = clave(df_incidencias, "fuente")

no_detectados = set_esperado - set_detectado
logger.info(f"Errores inyectados: {len(set_esperado)} | detectados por el ETL: "
            f"{len(set_esperado & set_detectado)} | no detectados: {len(no_detectados)}")
assert not no_detectados, f"Errores no detectados: {no_detectados}"
logger.info(f"=== Fin capa Silver | lote {ID_LOTE} ===")

2026-10-25 21:19:30 | INFO    | silver | Errores inyectados: 75 | detectados por el ETL: 75 | no detectados: 0
2026-10-25 21:19:30 | INFO    | silver | === Fin capa Silver | lote 1 ===
